In [1]:
import pandas as pd

df = pd.read_csv("../../data/processed/netflix_cleaned.csv")
df["country"].dropna().head(10)

0     United States
1            France
2     United States
3            Brazil
4     United States
5    United Kingdom
6     United States
7             India
8           Germany
9             India
Name: country, dtype: str

### Top 10 Countries by Netflix Content
Identify the countries with the highest number of Netflix titles.

In [2]:
countries = df["country"].dropna().str.split(",").explode().str.strip()
countries.value_counts().head(10)

country
United States     3240
India             1057
United Kingdom     638
Pakistan           421
Not Given          287
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Name: count, dtype: int64

In [3]:
import plotly.express as px

top_countries = countries.value_counts().head(10).reset_index()
top_countries.columns = ["Country", "Titles"]

fig = px.bar(
    top_countries.sort_values("Titles"),
    x="Titles", y="Country",
    orientation="h",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    height=550, width=850,
    title="Top 10 Countries by Netflix Content"
)

fig.update_traces(texttemplate="%{x:,}", textposition="outside")
fig.write_html("../outputs/top_10_countries.html")
fig.show()

In [4]:
country_counts = countries[countries != "Not Given"].value_counts().head(8).reset_index()
country_counts.columns = ["Country", "Titles"]

fig = px.pie(
    country_counts, names="Country", values="Titles",
    hole=0.5,
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark", height=500, width=700,
    title="Top 8 Countries — Netflix Content Share"
)

fig.update_traces(textposition="inside", textinfo="percent+label")
fig.write_html("../outputs/country_content_donut.html")
fig.show()

In [5]:
fig = px.treemap(
    country_counts, path=["Country"], values="Titles",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark", height=500, width=800,
    title="Netflix Content by Country"
)

fig.write_html("../outputs/country_content_treemap.html")
fig.show()

In [6]:
country_type = df.dropna(subset=["country"]).assign(
    country=df["country"].str.split(",")
).explode("country")

country_type["country"] = country_type["country"].str.strip()
top8 = country_type.loc[
    country_type["country"] != "Not Given", "country"
].value_counts().head(8).index

summary = country_type[
    country_type["country"].isin(top8)
].groupby(["country", "type"]).size().reset_index(name="Titles")

fig = px.bar(
    summary, x="country", y="Titles", color="type",
    barmode="stack",
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark", height=500, width=850,
    title="Movies vs TV Shows by Country"
)

fig.write_html("../outputs/country_type_comparison.html")
fig.show()

### Global Netflix Content Distribution
Explore the geographic distribution of Netflix titles across the top 20 countries.

In [7]:
map_data = countries[countries != "Not Given"].value_counts().head(20).reset_index()
map_data.columns = ["Country", "Titles"]

fig = px.choropleth(
    map_data,
    locations="Country",
    locationmode="country names",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Global Netflix Content Distribution",
    height=550,
    width=900
)

fig.write_html("../outputs/global_content_map.html")
fig.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_39192\138634583.py:4: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


### Country-Wise Business Insights
Identify the leading country and the diversity of Netflix content across countries.

In [8]:
country_counts = countries[countries != "Not Given"].value_counts()

print("Top Country:", country_counts.index[0])
print("Total Titles:", country_counts.iloc[0])
print("Countries Represented:", len(country_counts))

Top Country: United States
Total Titles: 3240
Countries Represented: 85
